# Excercise 1
1. Download the dataset `allegro/ConECT`
2. Use model `allegro/multislav-5lang` to translate part of this dataset from Czech to Polish with greedy decoding

In [1]:
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import GenerationConfig

In [2]:
model_id = "allegro/multislav-5lang"
dataset_id = "allegro/ConECT"
N_SAMPLES = 100
TRANSLATION_PROMPT = ">>pol<<{src_sent}"

In [3]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, dtype=torch.bfloat16,)
tokenizer = AutoTokenizer.from_pretrained(model_id)
dataset = load_dataset(dataset_id)

2025-10-17 21:55:02.224235: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [5]:
model.to(device)

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(80000, 1024, padding_idx=1)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(80000, 1024, padding_idx=1)
      (embed_positions): MarianSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_norm): LayerNorm((1024,), e

In [6]:
dataset = load_dataset(dataset_id)
# take N_SAMPLES samples from validset
dataset = dataset['validation']
dataset = dataset.select(range(N_SAMPLES))
# format prompt
dataset = dataset.map(lambda row: {'prompt': TRANSLATION_PROMPT.format(src_sent=row['cs_sent'])})

In [7]:
# encode, generate, decode

In [8]:
encodeds = tokenizer.batch_encode_plus(list(dataset['prompt']), return_tensors='pt', padding=True)

In [9]:
# greedy decoding
generation_config = GenerationConfig(do_sample=False)
outputs = model.generate(**encodeds.to(model.device), generation_config=generation_config)

In [10]:
translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(translations[0])

Buty sportowe męskie Big Star JJ174278 czarne 44


# Excercise 2
Evaluate translation with:
1. Lexical metrics: BLEU, chrF
2. Neural metrics: Comet (Unbabel/wmt22-comet-da), Comet QE (Unbabel/wmt20-comet-qe-da)

In [11]:
from sacrebleu import corpus_bleu, corpus_chrf
from comet import load_from_checkpoint, download_model

/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [12]:
COMET_MODEL_ID = "Unbabel/wmt22-comet-da"
COMET_QE_MODEL_ID = "Unbabel/wmt20-comet-qe-da"

In [13]:
# load comet models
comet_model = download_model(COMET_MODEL_ID)
comet_model = load_from_checkpoint(comet_model)

comet_qe_model = download_model(COMET_QE_MODEL_ID)
comet_qe_model = load_from_checkpoint(comet_qe_model)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/opt/conda/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt20-comet-qe-da/snapshots/2e7ffc84fb67d99cf92506611766463bb9230cfb/checkpoints/model.ckpt`


In [14]:
def get_metrics(src_list, trans_list, ref_list):
    """Calculate BELU, chrF, Comet and Comet QE, for list of source, translations and references"""
    bleu = corpus_bleu(trans_list, [[pl_sent] for pl_sent in ref_list])
    chrf = corpus_chrf(trans_list, [[pl_sent] for pl_sent in ref_list])
    
    comet_data = [{'src': src, 'mt': mt, 'ref': ref} for src, mt, ref in zip(src_list, trans_list, ref_list)] 
    comet_score = comet_model.predict(comet_data)
    comet_qe_score = comet_qe_model.predict(comet_data)
    
    return {'bleu':bleu.score, 'chrf':chrf.score, 'comet':comet_score.system_score, 'comet-qe':comet_qe_score.system_score}

In [15]:
eval_results = {}
eval_results['greedy'] = get_metrics(dataset['cs_sent'], translations, dataset['pl_sent'])

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 7/7 [00:01<00:00,  4.14it/s]
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.

In [16]:
eval_results

{'greedy': {'bleu': 18.575057999133602,
  'chrf': 77.41200331861435,
  'comet': 0.8184010171890259,
  'comet-qe': 0.09533200781792402}}

# Task 3
1. Generate translation hypotheses with epsilon sampling, N hypotheses for each source sentence (N=10, 50)

In [17]:
N_HYPS = 50
generation_config = GenerationConfig(
    do_sample=True, 
    epsilon_cutoff=0.02, 
    temperature=1.0, 
    num_return_sequences=N_HYPS, 
    max_new_tokens=128
)

In [18]:
# lets do it in batches (you can also use transformers pipeline, it supports batch_size)
batch_size = 8
hypotheses = []
for i in tqdm(range(0, len(dataset), batch_size)):
    # extract batch prompts
    batch_prompts = dataset['prompt'][i:i+batch_size]
    
    # encode, generate, decode
    encodeds = tokenizer.batch_encode_plus(batch_prompts, return_tensors='pt', padding=True)
    outputs = model.generate(**encodeds.to(model.device), generation_config=generation_config)
    decodeds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    hypotheses.extend(decodeds)

100%|██████████| 13/13 [00:42<00:00,  3.30s/it]


# Task 4
1. Decode candidates with MBR, use `mbrs` library
2. Evaluate translations with lexical and neural metrics

In [19]:
from mbrs.metrics import MetricCOMET, MetricCOMETkiwi
from mbrs.decoders import DecoderMBR, DecoderRerank

In [20]:
metric = MetricCOMET(MetricCOMET.Config(COMET_MODEL_ID, batch_size=8))
decoder = DecoderMBR(DecoderMBR.Config(), metric)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/opt/conda/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [21]:
mbr_translations = []
for i in tqdm(range(len(dataset))):
    hyps = hypotheses[i*N_HYPS: (i+1)*N_HYPS]
    translation = decoder.decode(hyps, hyps, dataset['cs_sent'][i])
    mbr_translations.append(translation.sentence[0])

100%|██████████| 100/100 [00:36<00:00,  2.71it/s]


In [22]:
len(mbr_translations) == len(dataset)

True

In [23]:
eval_results['mbr'] = get_metrics(dataset['cs_sent'], mbr_translations, dataset['pl_sent'])

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used

In [24]:
eval_results

{'greedy': {'bleu': 18.575057999133602,
  'chrf': 77.41200331861435,
  'comet': 0.8184010171890259,
  'comet-qe': 0.09533200781792402},
 'mbr': {'bleu': 44.17918226831576,
  'chrf': 83.98992171296409,
  'comet': 0.8595228338241577,
  'comet-qe': 0.19901370279490949}}

# Task 5
1. Decode candidates with QE reranking
2. Evaluate translations with lexical and neural metrics

In [25]:
from mbrs.metrics import MetricCOMETkiwi
from mbrs.decoders import DecoderRerank

In [26]:
config = MetricCOMETkiwi.Config(model=COMET_QE_MODEL_ID, batch_size=8)
qe_model = MetricCOMETkiwi(config)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt20-comet-qe-da/snapshots/2e7ffc84fb67d99cf92506611766463bb9230cfb/checkpoints/model.ckpt`
/opt/conda/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [27]:
decoder_config = DecoderRerank.Config()
decoder = DecoderRerank(config, qe_model)

In [28]:
qe_translations = []
for i in tqdm(range(len(dataset['cs_sent']))):
    current_hypotheses = hypotheses[i * N_HYPS: (i+1)*N_HYPS]
    current_translation = decoder.decode(current_hypotheses, dataset['cs_sent'][i]).sentence[0]
    qe_translations.append(current_translation)

100%|██████████| 100/100 [00:57<00:00,  1.74it/s]


In [29]:
eval_results['qe_reranking'] = get_metrics(dataset['cs_sent'], qe_translations, dataset['pl_sent'])

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used

In [30]:
import pandas as pd
df_results = pd.DataFrame(eval_results).T
print(df_results.round(4))

                 bleu     chrf   comet  comet-qe
greedy        18.5751  77.4120  0.8184    0.0953
mbr           44.1792  83.9899  0.8595    0.1990
qe_reranking  31.2394  83.9899  0.8637    0.3594
